# XGBoost — segunda versión (ratio 2:1, siguiendo el protocolo de `tuning/v3`)

## Configuración del tuning
- **Dataset:** ratio **2:1** pseudo-ausencias:presencias (equivale a "1 presencia por
  cada 2 pseudo-ausencias") — el ratio más representativo de la prevalencia real
  (~2.5%), reconstruido aquí con la MISMA lógica (misma tabla base `pixel_year_full.csv`,
  mismo buffer de exclusión de 3 km, misma semilla `random_state=42`) para que sea
  comparable cifra por cifra con las filas `2:1` de LR y RF ya reportadas en `tuning/v3`.
- **Búsqueda de hiperparámetros:** `RandomizedSearchCV` con **150 configuraciones**
  muestreadas al azar (de 300 combinaciones posibles), `GroupKFold(10)` sobre bloques
  espaciales de 0.25°, `scoring='average_precision'`, **restringido a `year<=2019`**
  (misma corrección de fuga temporal que v2/v3).
- **`scale_pos_weight` = n_negativos/n_positivos (≈2.0)**: con ratio 2:1 el dataset ya
  NO está balanceado (el doble de pseudo-ausencias que de presencias), así que se pondera
  la clase positiva para compensar — es el equivalente exacto del `class_weight='balanced'`
  que usaron LR y RF en 2:1, de modo que la comparación sea justa.

### Nota sobre la grilla pedida
La grilla original se pidió en términos de Random Forest (`n_estimators`, `max_depth`,
`max_features`, `min_samples_leaf`). XGBoost no tiene `max_features` ni
`min_samples_leaf` — se usan sus equivalentes nativos de boosting, preservando los
mismos valores donde es posible:

| Parámetro pedido (RF) | Valores pedidos | Equivalente en XGBoost | Valores usados |
|---|---|---|---|
| `n_estimators` | 100, 150, 200, 250, 350 | `n_estimators` (mismo nombre y valores) | 100, 150, 200, 250, 350 |
| `max_depth` | None, 10, 20, 30 | `max_depth` (0 = sin límite en XGBoost) | 0, 10, 20, 30 |
| `max_features` | 'sqrt', 0.5, 4 | `colsample_bytree` (fracción de columnas por árbol) | √9/9≈0.333, 0.5, 4/9≈0.444 |
| `min_samples_leaf` | 4, 6, 10, 20, 30 | `min_child_weight` (mínimo peso/muestras por hoja) | 4, 6, 10, 20, 30 |

$5 \times 4 \times 3 \times 5 = 300$ combinaciones posibles; se muestrean 150 al azar
(150 × 10 folds = 1,500 fits), una escala comparable a los 720 fits de RF en v1/v3.

## Aviso metodológico sobre el PR-AUC entre ratios
El PR-AUC de este dataset 2:1 **no es directamente comparable** con el del XGBoost 1:1,
porque la línea base del PR-AUC es la prevalencia de la clase positiva: 0.33 en 2:1
frente a 0.50 en 1:1. El PR-AUC en 2:1 se verá más bajo por construcción, NO por peor
desempeño. La comparación válida *entre ratios* se hace con AUC-ROC (invariante a la
prevalencia). La comparación *dentro* de este ratio (XGBoost vs. LR vs. RF, todos 2:1)
sí es válida en PR-AUC porque comparten la misma prevalencia.

## Qué hace este notebook
1. Reconstruye el dataset ratio 2:1 desde `pixel_year_full.csv` (idéntico a `tuning/v3`).
2. Afina XGBoost con `RandomizedSearchCV` (150 configuraciones, `year<=2019`).
3. Evalúa el modelo afinado con el protocolo completo (spatial block CV 10 folds +
   hold-out temporal `>=2020`, con matrices de confusión) — igual que LR y RF.
4. Guarda los resultados en esta carpeta (`model/xgboost/`): `xgboost_metrics_2to1.csv` y
   `xgboost_vs_lr_rf_comparison_2to1.csv` (comparación directa contra LR y RF en ratio 2:1).

In [8]:
# === train_xgboost.ipynb — Setup: dataset ratio 2:1 (mismo protocolo de tuning/v3) ===
# Reconstruye EXACTAMENTE el mismo dataset 2:1 que tuning/v3/tune_v3.ipynb generó para
# Random Forest y Logistic Regression (misma tabla base, mismo buffer de exclusión de
# 3 km, misma semilla random_state=42), para que XGBoost sea directamente comparable.
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              confusion_matrix)
from sklearn.base import clone

pixel_year = pd.read_csv('../../data/model_dataset/pixel_year_full.csv')
pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']
BLOCK = 0.25
EXCL_BUFFER = 3000            # 3 km — idéntico al usado en tuning/v3
buffer_deg = EXCL_BUFFER / 111000
RATIO = 2                     # 2 pseudo-ausencias por presencia (etiqueta de proyecto "2:1")
SEED = 42

presences = pixel_year[pixel_year['burned'] == 1].copy()
absences_all = pixel_year[pixel_year['burned'] == 0].copy()

kept_absences = []
for y in sorted(pixel_year['year'].unique()):
    pres_y = presences[presences['year'] == y][['lon','lat']].values
    abs_y  = absences_all[absences_all['year'] == y]
    if len(pres_y) == 0:
        kept_absences.append(abs_y); continue
    tree = cKDTree(pres_y)
    dists, _ = tree.query(abs_y[['lon','lat']].values, k=1)
    kept_absences.append(abs_y[dists > buffer_deg])
absences_far = pd.concat(kept_absences, ignore_index=True)

n_abs = min(len(absences_far), int(round(RATIO * len(presences))))
absences_sample = absences_far.sample(n=n_abs, random_state=SEED)
model_df = pd.concat([presences, absences_sample], ignore_index=True) \
             .sample(frac=1, random_state=SEED).reset_index(drop=True)
model_df['block'] = (model_df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                     (model_df['lat']//BLOCK).astype(int).astype(str)

n_pres = int(model_df['burned'].sum())
n_neg  = model_df.shape[0] - n_pres
print("Dataset ratio 2:1 ->", model_df.shape,
      f"({n_pres} presencias, {n_neg} pseudo-ausencias)")

Dataset ratio 2:1 -> (6231, 14) (2077 presencias, 4154 pseudo-ausencias)


In [9]:
# === RandomizedSearchCV — 150 configuraciones aleatorias, tuning SOLO con year<=2019 ===
tune_df = model_df[model_df['year'] <= 2019].copy()
X_tune = tune_df[pred_cols].values
y_tune = tune_df['burned'].astype(int).values
groups_tune = tune_df['block'].values
cv = GroupKFold(n_splits=10)

# Dataset 2:1 es imbalanceado -> pondera la clase positiva (equivale a class_weight='balanced')
SCALE_POS_WEIGHT = n_neg / n_pres
print(f"scale_pos_weight = n_neg/n_pos = {n_neg}/{n_pres} = {SCALE_POS_WEIGHT:.3f}")

param_distributions_xgb = {
    'n_estimators':     [100, 150, 200, 250, 350],
    'max_depth':        [0, 10, 20, 30],                 # 0 = sin límite (equivalente a None en RF)
    'colsample_bytree': [np.sqrt(len(pred_cols)) / len(pred_cols), 0.5, 4 / len(pred_cols)],
    'min_child_weight': [4, 6, 10, 20, 30],
}
n_combinations = (len(param_distributions_xgb['n_estimators']) *
                   len(param_distributions_xgb['max_depth']) *
                   len(param_distributions_xgb['colsample_bytree']) *
                   len(param_distributions_xgb['min_child_weight']))

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    scale_pos_weight=SCALE_POS_WEIGHT,     # dataset 2:1 imbalanceado -> corrección de clase
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_distributions_xgb,
    n_iter=150,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
xgb_search.fit(X_tune, y_tune, groups=groups_tune)

print(f"Grilla: {n_combinations} combinaciones posibles -> 150 muestreadas x 10 folds = "
      f"{150 * 10} fits")
print("Mejores hiperparámetros XGBoost:", xgb_search.best_params_)
print(f"Mejor PR-AUC (CV tuning, year<=2019): {xgb_search.best_score_:.3f}")

scale_pos_weight = n_neg/n_pos = 4154/2077 = 2.000
Grilla: 300 combinaciones posibles -> 150 muestreadas x 10 folds = 1500 fits
Mejores hiperparámetros XGBoost: {'n_estimators': 100, 'min_child_weight': 30, 'max_depth': 30, 'colsample_bytree': 0.4444444444444444}
Mejor PR-AUC (CV tuning, year<=2019): 0.777


In [10]:
def evaluate_model(name, estimator, model_df, pred_cols, n_splits=10, block=BLOCK):
    """Protocolo de validación IDÉNTICO a tuning/v3 (spatial block CV de 10 folds
    sobre el dataset completo + hold-out temporal train<=2019/test>=2020), con
    matrices de confusión — para que XGBoost sea directamente comparable con LR y RF."""
    X = model_df[pred_cols].values
    y = model_df['burned'].astype(int).values
    blocks = (model_df['lon']//block).astype(int).astype(str) + '_' + \
             (model_df['lat']//block).astype(int).astype(str)
    groups = blocks.values

    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    cm_spatial = np.zeros((2, 2), dtype=int)
    for tr, te in gkf.split(X, y, groups):
        m = clone(estimator).fit(X[tr], y[tr])
        prob = m.predict_proba(X[te])[:, 1]
        pred = m.predict(X[te])
        auc = roc_auc_score(y[te], prob)
        prauc = average_precision_score(y[te], prob)
        f1 = f1_score(y[te], pred)
        rows.append((auc, prauc, f1))
        cm_spatial += confusion_matrix(y[te], pred, labels=[0, 1])
    r = np.array(rows)
    print(f"  [{name}] SPATIAL  AUC={r[:,0].mean():.3f}±{r[:,0].std():.3f}  "
          f"PR-AUC={r[:,1].mean():.3f}±{r[:,1].std():.3f}  F1={r[:,2].mean():.3f}±{r[:,2].std():.3f}")

    tr = (model_df['year'] <= 2019).values
    te = (model_df['year'] >= 2020).values
    m = clone(estimator).fit(X[tr], y[tr])
    prob = m.predict_proba(X[te])[:, 1]
    pred = m.predict(X[te])
    cm_temporal = confusion_matrix(y[te], pred, labels=[0, 1])
    auc_t = roc_auc_score(y[te], prob)
    prauc_t = average_precision_score(y[te], prob)
    f1_t = f1_score(y[te], pred)
    print(f"  [{name}] TEMPORAL AUC={auc_t:.3f}  PR-AUC={prauc_t:.3f}  F1={f1_t:.3f}")

    return {
        'spatial': r.mean(axis=0), 'spatial_std': r.std(axis=0), 'spatial_cm': cm_spatial,
        'temporal': (auc_t, prauc_t, f1_t), 'temporal_cm': cm_temporal,
    }


def cm_to_dict(cm, prefix):
    tn, fp, fn, tp = cm.ravel()
    return {f'{prefix}_tn': int(tn), f'{prefix}_fp': int(fp),
            f'{prefix}_fn': int(fn), f'{prefix}_tp': int(tp)}


res_xgb = evaluate_model("XGBoost (tuned, ratio 2:1)", xgb_search.best_estimator_, model_df, pred_cols)

  [XGBoost (tuned, ratio 2:1)] SPATIAL  AUC=0.862±0.039  PR-AUC=0.745±0.073  F1=0.694±0.060
  [XGBoost (tuned, ratio 2:1)] TEMPORAL AUC=0.794  PR-AUC=0.540  F1=0.555


In [ ]:
# === Guardar resultados — mismo formato que tuning_v3_final_metrics.csv ===
row = {
    'ratio': '2:1',
    'model': 'XGBoost',
    'best_params': str(xgb_search.best_params_),
    'n_rows': model_df.shape[0],
    'n_presences': n_pres,
    'auc_spatial_mean': res_xgb['spatial'][0], 'auc_spatial_std': res_xgb['spatial_std'][0],
    'prauc_spatial_mean': res_xgb['spatial'][1], 'prauc_spatial_std': res_xgb['spatial_std'][1],
    'f1_spatial_mean': res_xgb['spatial'][2], 'f1_spatial_std': res_xgb['spatial_std'][2],
    'auc_temporal': res_xgb['temporal'][0], 'prauc_temporal': res_xgb['temporal'][1],
    'f1_temporal': res_xgb['temporal'][2],
}
row.update(cm_to_dict(res_xgb['spatial_cm'], 'cm_spatial'))
row.update(cm_to_dict(res_xgb['temporal_cm'], 'cm_temporal'))

xgb_metrics_df = pd.DataFrame([row])
xgb_metrics_df.to_csv('xgboost_metrics_2to1.csv', index=False)
print("Resultados guardados en: model/xgboost/xgboost_metrics_2to1.csv")
xgb_metrics_df[['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
                'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']]

Resultados guardados en: model/xgboost/xgboost_metrics_2to1.csv


,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,2:1,XGBoost,6231,2077,0.861716,0.74494,0.694382,0.793558,0.540112,0.554572


In [ ]:
# === Comparación directa contra LR y RF (mismo ratio 2:1, mismo protocolo de validación) ===
v3_sensitivity = pd.read_csv('../../tuning/v3/tuning_v3_sensitivity_metrics.csv')
cols = ['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
        'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']
lr_rf_21 = v3_sensitivity[v3_sensitivity['ratio'] == '2:1'][cols]

comparison_df = pd.concat([lr_rf_21, xgb_metrics_df[cols]], ignore_index=True)
comparison_df.to_csv('xgboost_vs_lr_rf_comparison_2to1.csv', index=False)
print("Comparación guardada en: model/xgboost/xgboost_vs_lr_rf_comparison_2to1.csv\n")
comparison_df

Comparación guardada en: model/xgboost/xgboost_vs_lr_rf_comparison_2to1.csv



,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,2:1,Logistic Regression,6231,2077,0.832055,0.702716,0.613300,0.808193,0.616809,0.557166
1,2:1,Random Forest,6231,2077,0.876874,0.779414,0.680450,0.819263,0.568723,0.560000
2,2:1,XGBoost,6231,2077,0.861716,0.744940,0.694382,0.793558,0.540112,0.554572


## Conclusiones

**Mejores hiperparámetros (`RandomizedSearchCV`, 150/300 combinaciones, tuning `year<=2019`): XGBoost: {'n_estimators': 100, 'min_child_weight': 30, 'max_depth': 30, 'colsample_bytree': 0.4444444444444444}
Mejor PR-AUC (CV tuning, year<=2019): 0.777

### Resultados (ratio 2:1, dataset idéntico al de LR/RF en `tuning/v3`)

| Modelo | AUC espacial | PR-AUC espacial | F1 espacial | AUC temporal | PR-AUC temporal | F1 temporal |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.83 | 0.70 | 0.61 | 0.81 | **0.62** | 0.56 |
| Random Forest | **0.88** | **0.78** | 0.68 | **0.82** | 0.57 | 0.56 |
| XGBoost | 0.86 | 0.74 | **0.69** | 0.79 | 0.54 | 0.55 |

(tabla completa en [`xgboost_vs_lr_rf_comparison_2to1.csv`](xgboost_vs_lr_rf_comparison_2to1.csv))

### Cómo leer estos números
- **PR-AUC en 2:1 vs 1:1:** el PR-AUC aquí se mide contra una línea base de prevalencia
  de **0.33** (no 0.50 como en 1:1). Por eso los valores 2:1 se ven más bajos que los del
  notebook 1:1 — es el efecto de prevalencia, NO peor desempeño. Para comparar *entre
  ratios* se usa AUC-ROC (invariante a la prevalencia); para comparar *dentro* de 2:1
  (XGBoost vs LR vs RF) el PR-AUC sí es válido porque comparten prevalencia.
- **Prioridad del proyecto = PR-AUC temporal** (el hold-out `>=2020` es la validación que
  más se parece a predecir un año futuro no observado, como el mapa 2026).

### ¿XGBoost mejora la predicción?

- **Validación espacial:** el orden es **Random Forest (PR-AUC 0.78) > XGBoost (0.74) >
  Logistic Regression (0.70)**. Los dos modelos de árboles superan al baseline lineal,
  confirmando que las relaciones no lineales aportan valor para predecir *dónde* arde.
  Sin embargo, XGBoost **no supera a Random Forest**: queda 0.04 por debajo en PR-AUC
  espacial, una diferencia comparable a la desviación entre folds.
- **Validación temporal (train ≤2019 / test ≥2020):** XGBoost obtiene el **PR-AUC temporal
  más bajo de los tres** (0.54, frente a 0.57 de RF y 0.62 de LR). A diferencia del ratio
  1:1 —donde XGBoost superaba levemente a RF (0.717 vs 0.704)—, en 2:1 esa ventaja
  **desaparece**. Esto muestra que la superioridad de XGBoost en 1:1 era **frágil y
  dependiente del ratio de muestreo**, no una mejora robusta del algoritmo.
- **La Regresión Logística obtiene el mejor PR-AUC temporal** (0.62), por encima de ambos
  modelos no lineales. Esto refuerza el hallazgo del proyecto: en la dimensión temporal la
  señal interanual es débil, y los modelos complejos ajustan ruido que no generaliza a
  años nuevos, mientras el baseline lineal —más rígido— generaliza mejor.

### Conclusión para el mapa de susceptibilidad 2026

El **Random Forest es el modelo final seleccionado**. Gana en discriminación espacial
(PR-AUC 0.78, AUC 0.88) —el objetivo directo del mapa de susceptibilidad—, es competitivo
en la validación temporal, y su desempeño es **robusto a la elección del ratio de
muestreo**, a diferencia de XGBoost, cuya ligera ventaja en 1:1 no se mantuvo en 2:1.
Además es más simple de interpretar y menos propenso a sobreajustar con pocos datos
(~6,231 filas). XGBoost queda como alternativa no lineal equivalente pero sin ventaja
demostrable; la Regresión Logística permanece como línea base transparente.

Esta comparación de tres modelos sobre un dataset y protocolo idénticos cierra la fase de
evaluación de modelos. Confirma dos conclusiones centrales del proyecto: (1) los modelos no
lineales mejoran la predicción **espacial** sobre el baseline, justificando el enfoque de
ML; y (2) el **ratio de muestreo influye más que la elección de algoritmo**, hasta el punto
de invertir el ranking entre RF y XGBoost. El siguiente paso es la interpretación del modelo
ganador mediante **SHAP + importancia por permutación**, seguido de la clasificación de
susceptibilidad (Jenks) y la generación del mapa 2026.